# Import Libraries

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

print("Libraries loaded.")

Libraries loaded.


# Import Artefacts from Task 1

In [5]:
# Load all artefacts saved during Task 1 training
# No retraining is performed — models are used exactly as trained

rf            = joblib.load('random_forest_model.pkl')
xgb           = joblib.load('xgboost_model.pkl')
label_encoder = joblib.load('label_encoder.pkl')
selector      = joblib.load('variance_selector.pkl')

print("Loaded artefacts:")

Loaded artefacts:


# Load Holdout Dataset

In [6]:
# bodmas_holdout.npz — formatted identically to the training dataset
# X: feature matrix (34435, 2381), y: binary labels 0=benign / 1=malware

data_h   = np.load('bodmas_holdout.npz')
X_holdout_raw = data_h['X']
y_binary_h    = data_h['y']

print(f"Holdout feature matrix : {X_holdout_raw.shape}")
print(f"Holdout binary labels  : {y_binary_h.shape}")
print(f"  Benign  (0)          : {(y_binary_h == 0).sum():,}")
print(f"  Malware (1)          : {(y_binary_h == 1).sum():,}")

Holdout feature matrix : (34435, 2381)
Holdout binary labels  : (34435,)
  Benign  (0)          : 18,363
  Malware (1)          : 16,072


# Load Holdout Metadata and Build Labels

In [7]:
metadata_h = pd.read_csv('bodmas_metadata_holdout.csv')
categories = pd.read_csv('bodmas_malware_category.csv')
categories = categories.rename(columns={'sha256': 'sha'})

metadata_h['binary_label'] = y_binary_h
merged_h = metadata_h[['sha']].merge(
    categories[['sha', 'category']], on='sha', how='left'
)
merged_h.loc[metadata_h['binary_label'].values == 0, 'category'] = 'benign'

y_labels_h = merged_h['category'].values

print("Holdout label distribution:")
print(pd.Series(y_labels_h).value_counts())
print(f"\nMissing labels: {pd.Series(y_labels_h).isna().sum()}")

Holdout label distribution:
benign                18363
trojan                 6724
worm                   6145
backdoor               2666
informationstealer      217
dropper                 135
ransomware              105
downloader               55
p2p-worm                 13
pua                       9
trojan-gamethief          2
cryptominer               1
Name: count, dtype: int64

Missing labels: 0


# Preprocess Holdout

In [8]:
X_holdout_raw = np.nan_to_num(X_holdout_raw, nan=0.0, posinf=0.0, neginf=0.0)

X_holdout = selector.transform(X_holdout_raw)
print(f"Feature before selection: {X_holdout_raw.shape[1]}")
print(f"Feature after selection : {X_holdout.shape[1]}")

y_encoded_h = label_encoder.transform(y_labels_h)
class_names = label_encoder.classes_
print(f"Encoded classes: ({len(class_names)}): {class_names}")
print(f"Holdout samples: {X_holdout.shape[0]:,}")

Feature before selection: 2381
Feature after selection : 1501
Encoded classes: (15): ['backdoor' 'benign' 'cryptominer' 'downloader' 'dropper' 'exploit'
 'informationstealer' 'p2p-worm' 'pua' 'ransomware' 'rootkit' 'trojan'
 'trojan-gamethief' 'virus' 'worm']
Holdout samples: 34,435


# Evaluation Function

# Discussion